In [ ]:
from pathlib import Path
from pprint import pprint

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
pd.read_pickle("../data/bus39/interim/lf.pkl").shape

(21783, 221)

In [ ]:
PROJECT_ROOT = Path().resolve().parent
REPORT_PATH = PROJECT_ROOT / "report-2026-05-29.joblib"

In [ ]:
data = joblib.load(REPORT_PATH)
len(data)

1044797

In [ ]:
pprint(data[0])

{'cct_true': 0.41,
 'cct_weighted_global': 0.6207310203411227,
 'cct_weighted_per_location': 0.4191452407446865,
 'crit_gen_true': 'g 03',
 'distance_mean': 1.9175723093230228,
 'distance_median': 1.93641569245816,
 'distance_min': 1.5678063615293918,
 'distance_norm': 0.8096434911340493,
 'distance_spread': 0.4654610286558878,
 'has_crit_gen_prediction': True,
 'has_location_prediction': True,
 'location_neighbor_count': 100,
 'location_true': 'bus 10',
 'location_weight_mass': 0.053943118425974663,
 'n_eff': 1834.6442285408953,
 'n_neighbors': 1853,
 'neighborhood_compactness': 0.12465052075947707,
 'prediction_summary': ReportSummary(cct_weighted=0.6207310203411227, cct_weighted_per_location={'Bus 04': 0.6843220689343971, 'Bus 05': 0.6013054473516604, 'Bus 06': 0.5528032598317759, 'Bus 07': 1.047210053260945, 'Bus 08': 0.8927672363657271, 'Bus 10': 0.4191452407446865, 'Bus 11': 0.4816790146215534, 'Bus 13': 0.4993173226606261, 'Bus 14': 0.6681282320683227, 'Line 03 - 04': 0.77242373

In [ ]:
def summary_value(summary, name: str, default=None):
    return getattr(summary, name, default) if summary is not None else default


df = pd.DataFrame(
    {
        "state": str(d["state"]),
        "cct_true": float(d["cct_true"]),
        "crit_gen_true": d["crit_gen_true"],
        "location_true": d["location_true"],
        "cct_weighted_per_location": d.get("cct_weighted_per_location"),
        "cct_weighted_global": d.get("cct_weighted_global", summary_value(d.get("prediction_summary"), "cct_weighted")),
        "has_crit_gen_prediction": d.get("has_crit_gen_prediction", d.get("prediction_summary") is not None),
        "has_location_prediction": d.get("has_location_prediction", d.get("cct_weighted_per_location") is not None),
        "location_weight_mass": d.get("location_weight_mass"),
        "location_neighbor_count": d.get("location_neighbor_count"),
        "n_neighbors": d.get("n_neighbors", summary_value(d.get("prediction_summary"), "n")),
        "n_eff": d.get("n_eff", summary_value(d.get("prediction_summary"), "n_eff")),
    }
    for d in data
)

df.head()

,state,cct_true,crit_gen_true,location_true,cct_weighted_per_location,cct_weighted_global,has_crit_gen_prediction,has_location_prediction,location_weight_mass,location_neighbor_count,n_neighbors,n_eff
0,11,0.41,g 03,bus 10,0.419145,0.620731,True,True,0.053943,100,1853,1834.644229
1,11,0.40,g 03,line 10 - 13,0.415757,0.620731,True,True,0.053943,100,1853,1834.644229
2,11,0.41,g 03,line 10 - 11,0.419818,0.620731,True,True,0.053943,100,1853,1834.644229
3,11,0.48,g 03,bus 13,0.499317,0.620731,True,True,0.053943,100,1853,1834.644229
4,11,0.48,g 03,line 13 - 14,0.500725,0.620731,True,True,0.053943,100,1853,1834.644229


In [ ]:
coverage = pd.Series(
    {
        "n_total": len(df),
        "n_with_crit_gen_prediction": int(df["has_crit_gen_prediction"].sum()),
        "n_with_location_prediction": int(df["has_location_prediction"].sum()),
        "crit_gen_coverage": float(df["has_crit_gen_prediction"].mean()),
        "location_coverage": float(df["has_location_prediction"].mean()),
        "n_missing_crit_gen_prediction": int((~df["has_crit_gen_prediction"]).sum()),
        "n_missing_location_prediction": int((~df["has_location_prediction"]).sum()),
    }
)

coverage

n_total                          1.044797e+06
n_with_crit_gen_prediction       1.044793e+06
n_with_location_prediction       1.044394e+06
crit_gen_coverage                9.999962e-01
location_coverage                9.996143e-01
n_missing_crit_gen_prediction    4.000000e+00
n_missing_location_prediction    4.030000e+02
dtype: float64

In [ ]:
missing = df.loc[~df["has_location_prediction"]].copy()
missing_by_crit_gen = missing["crit_gen_true"].value_counts()
missing_by_location = missing["location_true"].value_counts().head(20)

display(missing_by_crit_gen)
display(missing_by_location)

crit_gen_true
g 06    144
g 04     86
g 07     82
g 09     36
g 08     32
g 03     17
g 10      6
Name: count, dtype: int64

location_true
line 16 - 24    60
bus 24          47
line 15 - 16    34
bus 15          32
bus 16          24
line 16 - 21    22
bus 21          18
line 16 - 17    18
line 14 - 15    17
line 21 - 22    17
bus 02          12
line 02 - 03    10
line 01 - 02    10
bus 22          10
line 03 - 04     8
line 17 - 18     8
line 25 - 26     8
bus 17           7
bus 25           7
line 23 - 24     5
Name: count, dtype: int64

In [ ]:
def regression_metrics(frame: pd.DataFrame, pred_col: str) -> pd.Series:
    valid = frame[pred_col].notna()
    n_valid = int(valid.sum())

    if n_valid == 0:
        return pd.Series(
            {
                "n": 0,
                "coverage": 0.0,
                "mse": np.nan,
                "rmse": np.nan,
                "mae": np.nan,
                "abs_err_min": np.nan,
                "abs_err_q25": np.nan,
                "abs_err_q50": np.nan,
                "abs_err_q75": np.nan,
                "abs_err_q90": np.nan,
                "abs_err_q95": np.nan,
                "abs_err_q99": np.nan,
                "abs_err_max": np.nan,
            }
        )

    y_true = frame.loc[valid, "cct_true"].to_numpy(dtype=float)
    y_pred = frame.loc[valid, pred_col].to_numpy(dtype=float)
    err = np.abs(y_pred - y_true)

    return pd.Series(
        {
            "n": n_valid,
            "coverage": float(valid.mean()),
            "mse": mean_squared_error(y_true, y_pred),
            "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
            "mae": mean_absolute_error(y_true, y_pred),
            "abs_err_min": np.quantile(err, 0.00),
            "abs_err_q25": np.quantile(err, 0.25),
            "abs_err_q50": np.quantile(err, 0.50),
            "abs_err_q75": np.quantile(err, 0.75),
            "abs_err_q90": np.quantile(err, 0.90),
            "abs_err_q95": np.quantile(err, 0.95),
            "abs_err_q99": np.quantile(err, 0.99),
            "abs_err_max": np.quantile(err, 1.00),
        }
    )


metrics_conditional = pd.DataFrame(
    {
        "per_location": regression_metrics(df, "cct_weighted_per_location"),
        "global_fallback": regression_metrics(df, "cct_weighted_global"),
    }
)

df["cct_pred_overall"] = df["cct_weighted_per_location"].fillna(df["cct_weighted_global"])
metrics_overall = pd.DataFrame(
    {
        "location_then_global": regression_metrics(df, "cct_pred_overall"),
    }
)

metrics_conditional, metrics_overall

(             per_location  global_fallback
 n            1.044394e+06     1.044793e+06
 coverage     9.996143e-01     9.999962e-01
 mse          8.367947e-04     1.675612e-02
 rmse         2.892740e-02     1.294454e-01
 mae          1.759687e-02     9.082011e-02
 abs_err_min  0.000000e+00     3.226308e-07
 abs_err_q25  5.575477e-03     3.015416e-02
 abs_err_q50  1.206627e-02     6.913945e-02
 abs_err_q75  2.197544e-02     1.223477e-01
 abs_err_q90  3.589999e-02     1.847214e-01
 abs_err_q95  4.914215e-02     2.357132e-01
 abs_err_q99  1.066439e-01     4.675123e-01
 abs_err_max  7.895555e-01     1.333293e+00,
              location_then_global
 n                    1.044793e+06
 coverage             9.999962e-01
 mse                  8.491901e-04
 rmse                 2.914087e-02
 mae                  1.762983e-02
 abs_err_min          0.000000e+00
 abs_err_q25          5.577392e-03
 abs_err_q50          1.207028e-02
 abs_err_q75          2.198698e-02
 abs_err_q90          3.593648e-0

In [ ]:
by_crit_gen = (
    df.groupby("crit_gen_true", observed=True)
    .apply(lambda g: regression_metrics(g, "cct_weighted_per_location"), include_groups=False)
    .sort_values("mae")
    .astype({"n": int})
)

by_crit_gen.to_csv("./bus39_report.csv")

by_crit_gen

,n,coverage,mse,rmse,mae,abs_err_min,abs_err_q25,abs_err_q50,abs_err_q75,abs_err_q90,abs_err_q95,abs_err_q99,abs_err_max
crit_gen_true,,,,,,,,,,,,,
g 04,50842,0.998311,0.000231,0.015215,0.010880,0.000000,0.004130,0.008641,0.014963,0.022082,0.027451,0.042186,0.417939
g 07,135791,0.999396,0.000325,0.018027,0.013245,0.000000,0.004917,0.010535,0.018476,0.027401,0.033865,0.050168,0.410619
g 06,96645,0.998512,0.000319,0.017863,0.013247,0.000000,0.005082,0.010810,0.018602,0.027224,0.033265,0.046721,0.428873
g 09,325510,0.999889,0.000371,0.019250,0.013529,0.000000,0.004748,0.010151,0.018099,0.028320,0.037079,0.062535,0.615180
g 05,21720,1.000000,0.000919,0.030309,0.022654,0.000003,0.008601,0.018167,0.031454,0.046504,0.057254,0.086663,0.511784
g 08,54450,0.999413,0.001762,0.041976,0.023128,0.000000,0.005481,0.012122,0.023492,0.056041,0.092473,0.177377,0.509432
g 03,357557,0.999952,0.001513,0.038894,0.023726,0.000000,0.007502,0.016198,0.029321,0.047740,0.067035,0.149816,0.789556
g 10,1879,0.996817,0.005211,0.072188,0.057305,0.000050,0.022535,0.047739,0.082368,0.118287,0.137152,0.194050,0.306896


In [ ]:
missing_diagnostics = {
    "missing_by_crit_gen": df.loc[~df["has_crit_gen_prediction"], "crit_gen_true"].value_counts().head(20),
    "missing_by_location": df.loc[~df["has_location_prediction"], "location_true"].value_counts().head(20),
}

missing_diagnostics

{'missing_by_crit_gen': crit_gen_true
 g 10    4
 Name: count, dtype: int64,
 'missing_by_location': location_true
 line 16 - 24    60
 bus 24          47
 line 15 - 16    34
 bus 15          32
 bus 16          24
 line 16 - 21    22
 bus 21          18
 line 16 - 17    18
 line 14 - 15    17
 line 21 - 22    17
 bus 02          12
 line 02 - 03    10
 line 01 - 02    10
 bus 22          10
 line 03 - 04     8
 line 17 - 18     8
 line 25 - 26     8
 bus 17           7
 bus 25           7
 line 23 - 24     5
 Name: count, dtype: int64}

In [ ]:
df.columns

Index(['state', 'cct_true', 'crit_gen_true', 'location_true',
       'cct_weighted_per_location', 'cct_weighted_global',
       'has_crit_gen_prediction', 'has_location_prediction',
       'location_weight_mass', 'location_neighbor_count', 'n_neighbors',
       'n_eff', 'cct_pred_overall'],
      dtype='str')

In [ ]:
from sklearn.metrics import r2_score


def demo():
    valid = df["cct_weighted_per_location"].notna()

    subset = df.loc[valid]

    y_true = subset.loc[valid, "cct_true"].to_numpy(dtype=float)
    y_pred = subset.loc[valid, "cct_weighted_per_location"].to_numpy(dtype=float)

    subset["err"] = y_pred - y_true

    return subset["err"].corr(subset["location_weight_mass"], "spearman")

    # return r2_score(subset["location_weight_mass"], subset["err"])


demo()

np.float64(-0.03475749315309214)

In [ ]:
data[0]

{'state': 11,
 'state_norm': '11',
 'cct_true': 0.41,
 'crit_gen_true': 'g 03',
 'location_true': 'bus 10',
 'prediction_summary': ReportSummary(cct_weighted=0.6207310203411227, cct_weighted_per_location={'Bus 04': 0.6843220689343971, 'Bus 05': 0.6013054473516604, 'Bus 06': 0.5528032598317759, 'Bus 07': 1.047210053260945, 'Bus 08': 0.8927672363657271, 'Bus 10': 0.4191452407446865, 'Bus 11': 0.4816790146215534, 'Bus 13': 0.4993173226606261, 'Bus 14': 0.6681282320683227, 'Line 03 - 04': 0.772423737318097, 'Line 04 - 05': 0.6358314373182129, 'Line 04 - 14': 0.7020489445139961, 'Line 05 - 06': 0.557936150282245, 'Line 05 - 08': 0.6614899991638727, 'Line 06 - 07': 0.6001637754699014, 'Line 06 - 11': 0.4866099895968287, 'Line 07 - 08': 1.0691015818996383, 'Line 08 - 09': 1.52, 'Line 10 - 11': 0.4198176344658427, 'Line 10 - 13': 0.41575674777541893, 'Line 13 - 14': 0.5007254380829704, 'Line 14 - 15': 0.7668387141858266}, location_weight_mass={'Bus 04': 0.01564662825839845, 'Bus 05': 0.0502631

In [ ]:
from scipy.stats import spearmanr


def analysis():

    _df = pd.DataFrame(data)
    _df = _df.drop(columns=["prediction_summary"])
    _df = _df.dropna(subset=["cct_weighted_per_location"])
    _df["err"] = (_df["cct_true"] - _df["cct_weighted_per_location"]).abs()

    rho, p_value = spearmanr(_df.err, _df.n_eff)
    print("n_eff", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.neighborhood_compactness)
    print("neighborhood_compactness", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.n_neighbors)
    print("n_neighbors", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.location_weight_mass)
    print("location_weight_mass", f"{rho=} {p_value=}")

    dist_cols = [c for c in _df.columns if c.startswith("distance")]

    for col in dist_cols:
        rho, p_value = spearmanr(_df.err, _df[col])
        print(col, f"{rho=} {p_value=}")


analysis()

n_eff rho=np.float64(0.09553832946239892) p_value=np.float64(0.0)
neighborhood_compactness rho=np.float64(0.07753141456323542) p_value=np.float64(0.0)
n_neighbors rho=np.float64(0.09045006224724746) p_value=np.float64(0.0)
location_weight_mass rho=np.float64(-0.10414016325868802) p_value=np.float64(0.0)
distance_min rho=np.float64(-0.050756834197032585) p_value=np.float64(0.0)
distance_mean rho=np.float64(-0.05587021722842067) p_value=np.float64(0.0)
distance_median rho=np.float64(-0.056124030725831574) p_value=np.float64(0.0)
distance_spread rho=np.float64(-0.05702055531910454) p_value=np.float64(0.0)
distance_norm rho=np.float64(0.014583538913141009) p_value=np.float64(3.080674795417717e-50)


In [ ]:
_df = pd.DataFrame(data)
_df = _df.drop(columns=["prediction_summary"])
_df = _df.dropna(subset=["cct_weighted_per_location"])
_df["err"] = (_df["cct_true"] - _df["cct_weighted_per_location"]).abs()

In [ ]:
import pandas as pd


def risk_coverage(df, metric, higher_is_better=True, coverages=(1.0, 0.95, 0.9, 0.8, 0.7, 0.5)):

    x = df.dropna(subset=[metric, "err"]).copy()
    x = x.sort_values(metric, ascending=not higher_is_better)

    rows = []
    n = len(x)
    for cov in coverages:
        k = int(np.ceil(cov * n))
        kept = x.iloc[:k]
        rows.append(
            {
                "metric": metric,
                "coverage": cov,
                "n": k,
                "mae": kept["err"].mean(),
                "rmse": np.sqrt((kept["err"] ** 2).mean()),
                "q90": kept["err"].quantile(0.90),
                "q95": kept["err"].quantile(0.95),
            }
        )
    return pd.DataFrame(rows)


metrics = {
    "location_weight_mass": True,
    "n_eff": True,
    "n_neighbors": True,
    "neighborhood_compactness": True,
    "distance_min": False,
    "distance_mean": False,
    "distance_median": False,
    "distance_spread": False,
    "distance_norm": False,
}

out = []
for metric, higher_is_better in metrics.items():
    out.append(risk_coverage(_df, metric, higher_is_better))

rc = pd.concat(out, ignore_index=True)
rc.to_csv("./risk_coverage_bus39.csv")
rc

,metric,coverage,n,mae,rmse,q90,q95
0,location_weight_mass,1.00,1044394,0.017597,0.028927,0.035900,0.049142
1,location_weight_mass,0.95,992175,0.016730,0.025696,0.034688,0.046474
2,location_weight_mass,0.90,939955,0.016543,0.025531,0.034181,0.045839
3,location_weight_mass,0.80,835516,0.016078,0.025253,0.032856,0.044096
4,location_weight_mass,0.70,731076,0.015240,0.023243,0.031258,0.041286
5,location_weight_mass,0.50,522197,0.015407,0.023663,0.031279,0.041101
6,n_eff,1.00,1044394,0.017597,0.028927,0.035900,0.049142
7,n_eff,0.95,992175,0.017519,0.028734,0.035699,0.048819
8,n_eff,0.90,939955,0.017705,0.029059,0.036013,0.049287
9,n_eff,0.80,835516,0.017698,0.028872,0.036216,0.049101


## ELES data


In [ ]:
pd.read_pickle("../data/eles-2026-01/interim/lf.pkl").shape

(4402, 10298)

In [ ]:
PROJECT_ROOT = Path().resolve().parent
REPORT_PATH = PROJECT_ROOT / "report-2026-06-10-eles.joblib"
assert REPORT_PATH.exists() and REPORT_PATH.is_file()

data = joblib.load(REPORT_PATH)
len(data)

323627

In [ ]:
pprint(data[0])

{'cct_true': 0.85,
 'cct_weighted_global': 0.6941863234390536,
 'cct_weighted_per_location': 0.8714490845599662,
 'crit_gen_true': '28w-g-000000038l',
 'distance_mean': 181.21753443349266,
 'distance_median': 198.32439675813873,
 'distance_min': 9.494912485006276,
 'distance_norm': 0.04787566552684639,
 'distance_spread': 258.4635697704325,
 'has_crit_gen_prediction': True,
 'has_location_prediction': True,
 'location_neighbor_count': 60,
 'location_true': '28t-0000000109-h',
 'location_weight_mass': 0.14285714285714285,
 'n_eff': 14.568301487795992,
 'n_neighbors': 398,
 'neighborhood_compactness': 0.012065267173612537,
 'prediction_summary': ReportSummary(cct_weighted=0.6941863234390536, cct_weighted_per_location={'28T-0000000109-H': 0.8714490845599662, '28T-0000000110-3': 0.7388262195891847, '28T-0000000111-0': 0.7388262195891847, '28T-0000000112-Y': 0.7445507618691627, '28T-0000000113-V': 0.7299999999999646, '28T-0000000117-J': 0.5151013701675566, '28T-0000000118-G': 0.520550608298

In [ ]:
def summary_value(summary, name: str, default=None):
    return getattr(summary, name, default) if summary is not None else default


df = pd.DataFrame(
    {
        "state": str(d["state"]),
        "cct_true": float(d["cct_true"]),
        "crit_gen_true": d["crit_gen_true"],
        "location_true": d["location_true"],
        "cct_weighted_per_location": d.get("cct_weighted_per_location"),
        "cct_weighted_global": d.get("cct_weighted_global", summary_value(d.get("prediction_summary"), "cct_weighted")),
        "has_crit_gen_prediction": d.get("has_crit_gen_prediction", d.get("prediction_summary") is not None),
        "has_location_prediction": d.get("has_location_prediction", d.get("cct_weighted_per_location") is not None),
        "location_weight_mass": d.get("location_weight_mass"),
        "location_neighbor_count": d.get("location_neighbor_count"),
        "n_neighbors": d.get("n_neighbors", summary_value(d.get("prediction_summary"), "n")),
        "n_eff": d.get("n_eff", summary_value(d.get("prediction_summary"), "n_eff")),
    }
    for d in data
)

df.head()

,state,cct_true,crit_gen_true,location_true,cct_weighted_per_location,cct_weighted_global,has_crit_gen_prediction,has_location_prediction,location_weight_mass,location_neighbor_count,n_neighbors,n_eff
0,0_4,0.85,28w-g-000000038l,28t-0000000109-h,0.871449,0.694186,True,True,0.142857,60,398,14.568301
1,0_4,0.28,28w-g-000000068c,28t-0000000186-v,0.300000,0.286667,True,True,0.333333,7,15,3.000000
2,0_4,0.28,28w-g-000000068c,28t-0000000186-v,0.300000,0.286667,True,True,0.333333,7,15,3.000000
3,0_4,0.52,28w-g-000000038l,28t-0000000117-j,0.515101,0.694186,True,True,0.142857,62,398,14.568301
4,0_4,0.29,28w-g-0000000203,28t000000000301k,0.340000,0.296416,True,True,0.018661,20,416,3.600414


In [ ]:
coverage = pd.Series(
    {
        "n_total": len(df),
        "n_with_crit_gen_prediction": int(df["has_crit_gen_prediction"].sum()),
        "n_with_location_prediction": int(df["has_location_prediction"].sum()),
        "crit_gen_coverage": float(df["has_crit_gen_prediction"].mean()),
        "location_coverage": float(df["has_location_prediction"].mean()),
        "n_missing_crit_gen_prediction": int((~df["has_crit_gen_prediction"]).sum()),
        "n_missing_location_prediction": int((~df["has_location_prediction"]).sum()),
    }
)

coverage

n_total                          323627.000000
n_with_crit_gen_prediction       323458.000000
n_with_location_prediction       321804.000000
crit_gen_coverage                     0.999478
location_coverage                     0.994367
n_missing_crit_gen_prediction       169.000000
n_missing_location_prediction      1823.000000
dtype: float64

In [ ]:
missing = df.loc[~df["has_location_prediction"]].copy()
missing_by_crit_gen = missing["crit_gen_true"].value_counts()
missing_by_location = missing["location_true"].value_counts().head(20)

display(missing_by_crit_gen)
display(missing_by_location)

crit_gen_true
28w-g-000000042u    119
28w-g-000000035r     97
28w-g-000000043s     72
28w-g-000000015x     63
28w-g-000000017t     61
                   ... 
28w-g-000000058f      3
28w-g-000000055l      2
28w-g-0000000009      2
28w000000000008d      2
sym62741              1
Name: count, Length: 68, dtype: int64

location_true
28t000000000301k    79
28t0000000009576    58
28t0000000009584    55
28t-0000000204-r    51
28t-0000000005-y    51
28t-0000000077-0    51
28t-0000000134-k    50
28t-0000000212-t    49
28t-0000000149-1    49
28t-0000000125-l    48
28t0000000009592    47
28t-0000000056-b    46
28t-0000000021-1    46
28t000000000960h    46
28t-0000000102-1    46
28t-0000000004-0    45
28t000000000955a    44
28t-0000000103-z    43
28t-0000000126-i    42
28t-0000000053-k    42
Name: count, dtype: int64

In [ ]:
def regression_metrics(frame: pd.DataFrame, pred_col: str) -> pd.Series:
    valid = frame[pred_col].notna()
    n_valid = int(valid.sum())

    if n_valid == 0:
        return pd.Series(
            {
                "n": 0,
                "coverage": 0.0,
                "mse": np.nan,
                "rmse": np.nan,
                "mae": np.nan,
                "abs_err_min": np.nan,
                "abs_err_q25": np.nan,
                "abs_err_q50": np.nan,
                "abs_err_q75": np.nan,
                "abs_err_q90": np.nan,
                "abs_err_q95": np.nan,
                "abs_err_q99": np.nan,
                "abs_err_max": np.nan,
            }
        )

    y_true = frame.loc[valid, "cct_true"].to_numpy(dtype=float)
    y_pred = frame.loc[valid, pred_col].to_numpy(dtype=float)
    err = np.abs(y_pred - y_true)

    return pd.Series(
        {
            "n": n_valid,
            "coverage": float(valid.mean()),
            "mse": mean_squared_error(y_true, y_pred),
            "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
            "mae": mean_absolute_error(y_true, y_pred),
            "abs_err_min": np.quantile(err, 0.00),
            "abs_err_q25": np.quantile(err, 0.25),
            "abs_err_q50": np.quantile(err, 0.50),
            "abs_err_q75": np.quantile(err, 0.75),
            "abs_err_q90": np.quantile(err, 0.90),
            "abs_err_q95": np.quantile(err, 0.95),
            "abs_err_q99": np.quantile(err, 0.99),
            "abs_err_max": np.quantile(err, 1.00),
        }
    )


metrics_conditional = pd.DataFrame(
    {
        "per_location": regression_metrics(df, "cct_weighted_per_location"),
        "global_fallback": regression_metrics(df, "cct_weighted_global"),
    }
)

df["cct_pred_overall"] = df["cct_weighted_per_location"].fillna(df["cct_weighted_global"])
metrics_overall = pd.DataFrame(
    {
        "location_then_global": regression_metrics(df, "cct_pred_overall"),
    }
)

metrics_conditional, metrics_overall

(              per_location  global_fallback
 n            321804.000000    323458.000000
 coverage          0.994367         0.999478
 mse               0.012515         0.020034
 rmse              0.111870         0.141542
 mae               0.048822         0.080550
 abs_err_min       0.000000         0.000000
 abs_err_q25       0.000002         0.018000
 abs_err_q50       0.010000         0.041438
 abs_err_q75       0.050000         0.093333
 abs_err_q90       0.130000         0.190980
 abs_err_q95       0.210000         0.295000
 abs_err_q99       0.530000         0.608265
 abs_err_max       1.400000         1.726563,
              location_then_global
 n                   323458.000000
 coverage                 0.999478
 mse                      0.012949
 rmse                     0.113795
 mae                      0.049607
 abs_err_min              0.000000
 abs_err_q25              0.000003
 abs_err_q50              0.010000
 abs_err_q75              0.050000
 abs_err_q90       

In [ ]:
by_crit_gen = (
    df.groupby("crit_gen_true", observed=True)
    .apply(lambda g: regression_metrics(g, "cct_weighted_per_location"), include_groups=False)
    .sort_values("mae")
    .astype({"n": int})
)

by_crit_gen.to_csv("./eles2026-01_report.csv")

by_crit_gen

,n,coverage,mse,rmse,mae,abs_err_min,abs_err_q25,abs_err_q50,abs_err_q75,abs_err_q90,abs_err_q95,abs_err_q99,abs_err_max
crit_gen_true,,,,,,,,,,,,,
28w-g-000000052r,42,0.840000,0.000033,0.005748,0.003203,0.000000,0.000000e+00,2.031902e-12,0.010000,0.010000,0.010000,0.012172,0.013682
28w0000000000384,2114,1.000000,0.000208,0.014423,0.004140,0.000000,0.000000e+00,9.936496e-15,0.009488,0.010000,0.010005,0.040000,0.350000
28w-g-000000057h,251,0.976654,0.000092,0.009616,0.005921,0.000000,5.551115e-17,1.145716e-09,0.010000,0.020000,0.020000,0.030000,0.030000
28w0000000000376,1732,0.997121,0.001951,0.044169,0.006433,0.000000,0.000000e+00,9.747758e-14,0.009968,0.010000,0.013530,0.030000,0.799915
28w-g-000000058f,8003,0.999625,0.000258,0.016058,0.007900,0.000000,3.330669e-16,9.728192e-04,0.010000,0.020000,0.030000,0.070000,0.228345
...,...,...,...,...,...,...,...,...,...,...,...,...,...
28w-g-000000023y,6811,0.995615,0.056479,0.237653,0.133090,0.000000,1.000000e-02,5.177768e-02,0.170000,0.370000,0.584178,0.909828,1.380000
sym79282,12,1.000000,0.025332,0.159161,0.134512,0.019869,7.023755e-02,1.100000e-01,0.222565,0.238705,0.239742,0.239863,0.239894
28w-g-000000049g,562,0.938230,0.074374,0.272716,0.145876,0.000000,1.000000e-02,4.000000e-02,0.170000,0.539000,0.723115,0.940000,1.019730


In [ ]:
missing_diagnostics = {
    "missing_by_crit_gen": df.loc[~df["has_crit_gen_prediction"], "crit_gen_true"].value_counts().head(20),
    "missing_by_location": df.loc[~df["has_location_prediction"], "location_true"].value_counts().head(20),
}

missing_diagnostics

{'missing_by_crit_gen': crit_gen_true
 28w-g-000000017t    23
 28w-g-000000016v    22
 28w-g-000000049g    15
 28w-g-000000015x    14
 28w-g-000000035r    12
 28w-g-000000052r     8
 28w-g-000000033v     8
 28w-g-000000023y     8
 28w-g-000000048i     6
 28w-g-000000014z     6
 28w-g-000000057h     6
 28w-g-000000019p     5
 28w-g-000000046m     5
 sym88882             5
 28w-g-0000000130     5
 28w-g-000000070p     4
 28w-g-000000040y     3
 28w-g-000000026s     3
 28w-g-0000000017     2
 28w-g-000000034t     2
 Name: count, dtype: int64,
 'missing_by_location': location_true
 28t000000000301k    79
 28t0000000009576    58
 28t0000000009584    55
 28t-0000000204-r    51
 28t-0000000005-y    51
 28t-0000000077-0    51
 28t-0000000134-k    50
 28t-0000000212-t    49
 28t-0000000149-1    49
 28t-0000000125-l    48
 28t0000000009592    47
 28t-0000000056-b    46
 28t-0000000021-1    46
 28t000000000960h    46
 28t-0000000102-1    46
 28t-0000000004-0    45
 28t000000000955a    44
 28t-000

In [ ]:
df.columns

Index(['state', 'cct_true', 'crit_gen_true', 'location_true',
       'cct_weighted_per_location', 'cct_weighted_global',
       'has_crit_gen_prediction', 'has_location_prediction',
       'location_weight_mass', 'location_neighbor_count', 'n_neighbors',
       'n_eff', 'cct_pred_overall'],
      dtype='str')

In [ ]:
def demo():
    valid = df["cct_weighted_per_location"].notna()

    subset = df.loc[valid]

    y_true = subset.loc[valid, "cct_true"].to_numpy(dtype=float)
    y_pred = subset.loc[valid, "cct_weighted_per_location"].to_numpy(dtype=float)

    subset["err"] = y_pred - y_true

    return subset["err"].corr(subset["location_weight_mass"], "spearman")

    # return r2_score(subset["location_weight_mass"], subset["err"])


demo()

np.float64(0.04526873318272704)

In [ ]:
def analysis():

    _df = pd.DataFrame(data)
    _df = _df.drop(columns=["prediction_summary"])
    _df = _df.dropna(subset=["cct_weighted_per_location"])
    _df["err"] = (_df["cct_true"] - _df["cct_weighted_per_location"]).abs()

    rho, p_value = spearmanr(_df.err, _df.n_eff)
    print("n_eff", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.neighborhood_compactness)
    print("neighborhood_compactness", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.n_neighbors)
    print("n_neighbors", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.location_weight_mass)
    print("location_weight_mass", f"{rho=} {p_value=}")

    dist_cols = [c for c in _df.columns if c.startswith("distance")]

    for col in dist_cols:
        rho, p_value = spearmanr(_df.err, _df[col])
        print(col, f"{rho=} {p_value=}")


analysis()

n_eff rho=np.float64(-0.015086662432293708) p_value=np.float64(1.1406705194832942e-17)
neighborhood_compactness rho=nan p_value=nan
n_neighbors rho=np.float64(-0.09498960749097249) p_value=np.float64(0.0)
location_weight_mass rho=np.float64(-0.13435500935374167) p_value=np.float64(0.0)
distance_min rho=np.float64(0.25972118780488984) p_value=np.float64(0.0)
distance_mean rho=np.float64(0.054176371462077315) p_value=np.float64(1.0318323814124865e-207)
distance_median rho=np.float64(0.030841802501267647) p_value=np.float64(1.433651717711609e-68)
distance_spread rho=np.float64(-0.2384068520140008) p_value=np.float64(0.0)
distance_norm rho=np.float64(0.2723306925647721) p_value=np.float64(0.0)


In [ ]:
import pandas as pd

_df = pd.DataFrame(data)
_df = _df.drop(columns=["prediction_summary"])
_df = _df.dropna(subset=["cct_weighted_per_location"])
_df["err"] = (_df["cct_true"] - _df["cct_weighted_per_location"]).abs()


def risk_coverage(df, metric, higher_is_better=True, coverages=(1.0, 0.95, 0.9, 0.8, 0.7, 0.5)):

    x = df.dropna(subset=[metric, "err"]).copy()
    x = x.sort_values(metric, ascending=not higher_is_better)

    rows = []
    n = len(x)
    for cov in coverages:
        k = int(np.ceil(cov * n))
        kept = x.iloc[:k]
        rows.append(
            {
                "metric": metric,
                "coverage": cov,
                "n": k,
                "mae": kept["err"].mean(),
                "rmse": np.sqrt((kept["err"] ** 2).mean()),
                "q90": kept["err"].quantile(0.90),
                "q95": kept["err"].quantile(0.95),
            }
        )
    return pd.DataFrame(rows)


metrics = {
    "location_weight_mass": True,
    "n_eff": True,
    "n_neighbors": True,
    "neighborhood_compactness": True,
    "distance_min": False,
    "distance_mean": False,
    "distance_median": False,
    "distance_spread": False,
    "distance_norm": False,
}

out = []
for metric, higher_is_better in metrics.items():
    out.append(risk_coverage(_df, metric, higher_is_better))

rc = pd.concat(out, ignore_index=True)
rc.to_csv("./risk_coverage_eles2026-01.csv")

rc

,metric,coverage,n,mae,rmse,q90,q95
0,location_weight_mass,1.00,321804,0.048822,0.111870,0.130000,0.210000
1,location_weight_mass,0.95,305714,0.046464,0.108092,0.120000,0.200000
2,location_weight_mass,0.90,289624,0.044792,0.105408,0.120000,0.200000
3,location_weight_mass,0.80,257444,0.044261,0.105407,0.120000,0.200000
4,location_weight_mass,0.70,225263,0.042583,0.103594,0.110000,0.190000
5,location_weight_mass,0.50,160902,0.043879,0.108194,0.120000,0.200000
6,n_eff,1.00,321804,0.048822,0.111870,0.130000,0.210000
7,n_eff,0.95,305714,0.047984,0.110515,0.129997,0.210000
8,n_eff,0.90,289624,0.047202,0.109288,0.126495,0.209928
9,n_eff,0.80,257444,0.046731,0.107678,0.121641,0.200000
